# CNF metrics comparison for LNS vs whole-solve experiments

This notebook loads the experiment-level CSV logs from `lns_clean/logs`, builds per-map summaries for
CNF variables and clauses, and makes it easy to slice the data further by map and agent count.


## Data layout assumptions

* The log root lives at `lns_clean/logs` relative to the repository root.
* Inside `logs` there are two top-level folders: `LNS` and `Whole solve`.
* Within each of those folders there is a map-category directory that contains the individual map
  directories, each of which holds an `Experiment*.csv` file.

Each `Experiment*.csv` file is expected to contain at least the following columns:

* `experiment_id`
* `total cnf clauses`
* `total cnf variables`

The `experiment_id` encodes how many agents were used, so the notebook extracts an `agent_count`
column from it for optional grouping.


In [184]:
from pathlib import Path
import re
import pandas as pd

pd.options.display.float_format = "{:.2f}".format

LOG_ROOT = Path("../lns_clean/logs")
SCENARIO_FOLDERS = {
    "LNS": LOG_ROOT / "LNS",
    "Whole solve": LOG_ROOT / "WholeSolve",  # directory is named "WholeSolve" on disk
}
REQUIRED_COLUMNS = {"experiment_id", "total_cnf_clauses", "total_cnf_variables"}


In [185]:
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize column names to snake_case for easier access."""

    normalized = df.copy()
    normalized.columns = [col.strip().lower().replace(" ", "_") for col in normalized.columns]
    return normalized


def extract_agent_count(experiment_id: str):
    """Extract the agent count from an experiment identifier.

    Prefer the explicit "scen-agents<k>" pattern used in our experiment IDs,
    and fall back to a more generic heuristic only if that fails.
    """

    if not isinstance(experiment_id, str):
        return None

    # Primary pattern used in these logs: ..._scen-agents10-idx...
    match = re.search(r"scen-agents(\d+)", experiment_id)
    if match:
        return int(match.group(1))

    # Fallback: look for "<k> agents" text, if it ever appears
    match = re.search(r"(\d+)\s*agents?", experiment_id, flags=re.IGNORECASE)
    if match:
        return int(match.group(1))

    # Last-resort heuristic: take the last integer in the string
    numbers = re.findall(r"\d+", experiment_id)
    return int(numbers[-1]) if numbers else None


def load_experiment_data(scenario_name: str, scenario_path: Path) -> pd.DataFrame:
    """Load and annotate experiment-level CSV files for a single scenario."""

    if not scenario_path.exists():
        raise FileNotFoundError(
            f"Expected to find experiment logs at {scenario_path}. Adjust `LOG_ROOT` if needed."
        )

    csv_paths = list(scenario_path.rglob("experiments.csv"))
    if not csv_paths:
        raise FileNotFoundError(f"No Experiment*.csv files found under {scenario_path}.")

    frames = []
    for csv_path in csv_paths:
        raw = pd.read_csv(csv_path)
        df = normalize_columns(raw)

        missing = REQUIRED_COLUMNS - set(df.columns)
        if missing:
            raise KeyError(
                f"{csv_path} is missing required columns: {', '.join(sorted(missing))}"
            )

        df = df.assign(
            scenario=scenario_name,
            map_category=csv_path.parent.parent.name,
            map_name=csv_path.parent.name,
            csv_path=csv_path,
        )
        df["agent_count"] = df["experiment_id"].apply(extract_agent_count)
        frames.append(df)

    return pd.concat(frames, ignore_index=True)


In [186]:
lns_experiments = load_experiment_data("LNS", SCENARIO_FOLDERS["LNS"])
whole_experiments = load_experiment_data("Whole solve", SCENARIO_FOLDERS["Whole solve"])

# Identify experiments that never produced a CNF (0 clauses and 0 variables)
lns_zero_cnf = lns_experiments[
    (lns_experiments["total_cnf_clauses"] == 0)
    & (lns_experiments["total_cnf_variables"] == 0)
]
whole_zero_cnf = whole_experiments[
    (whole_experiments["total_cnf_clauses"] == 0)
    & (whole_experiments["total_cnf_variables"] == 0)
]

print("LNS experiments with 0 CNF clauses and 0 CNF variables:", len(lns_zero_cnf))
if not lns_zero_cnf.empty:
    display(
        lns_zero_cnf[
            [
                "experiment_id",
                "map_category",
                "map_name",
                "total_cnf_clauses",
                "total_cnf_variables",
            ]
        ]
    )

print("Whole-solve experiments with 0 CNF clauses and 0 CNF variables:", len(whole_zero_cnf))
if not whole_zero_cnf.empty:
    display(
        whole_zero_cnf[
            [
                "experiment_id",
                "map_category",
                "map_name",
                "total_cnf_clauses",
                "total_cnf_variables",
            ]
        ]
    )

# Filter them out before computing any summaries
lns_experiments = lns_experiments.drop(lns_zero_cnf.index)
whole_experiments = whole_experiments.drop(whole_zero_cnf.index)

all_experiments = pd.concat([lns_experiments, whole_experiments], ignore_index=True)

lns_experiments.head()


LNS experiments with 0 CNF clauses and 0 CNF variables: 411


,experiment_id,map_category,map_name,total_cnf_clauses,total_cnf_variables
0,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,LNS,Berlin,0,0
1,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,LNS,Berlin,0,0
2,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,LNS,Berlin,0,0
5,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,LNS,Berlin,0,0
6,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,LNS,Berlin,0,0
...,...,...,...,...,...
2558,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,LNS,Rooms,0,0
2576,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,LNS,Rooms,0,0
2581,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,LNS,Rooms,0,0
2605,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,LNS,Rooms,0,0


Whole-solve experiments with 0 CNF clauses and 0 CNF variables: 28


,experiment_id,map_category,map_name,total_cnf_clauses,total_cnf_variables
0,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,WholeSolve,Berlin,0,0
1,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,WholeSolve,Berlin,0,0
2,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,WholeSolve,Berlin,0,0
3,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,WholeSolve,Warehouse2-2,0,0
4,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,WholeSolve,Warehouse2-2,0,0
5,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,WholeSolve,Warehouse2-2,0,0
6,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,WholeSolve,Warehouse2-2,0,0
7,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,WholeSolve,Warehouse2-2,0,0
8,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,WholeSolve,Warehouse2-2,0,0
9,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,WholeSolve,Warehouse2-2,0,0


,experiment_id,map_path,scenario_path,num_agents,scenario_index,seed,solved,makespan_success,total_runtime_ms,total_cnf_clauses,...,total_mdd_build_ms,total_cnf_build_ms,total_lazy_wall_ms,total_lazy_solver_wall_ms,total_lazy_solver_reported_ms,scenario,map_category,map_name,csv_path,agent_count
3,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,/home/max/Documents/PhD/MAPF/LNS/MAPF_LNS_SAT/...,/home/max/Documents/PhD/MAPF/LNS/MAPF_LNS_SAT/...,10,3,42,1,424,1539.95,357,...,0.00,0.21,0.33,0.26,0.03,LNS,LNS,Berlin,../lns_clean/logs/LNS/Berlin/experiments.csv,10
4,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,/home/max/Documents/PhD/MAPF/LNS/MAPF_LNS_SAT/...,/home/max/Documents/PhD/MAPF/LNS/MAPF_LNS_SAT/...,10,4,42,1,427,1618.81,24,...,0.00,0.02,0.04,0.04,0.01,LNS,LNS,Berlin,../lns_clean/logs/LNS/Berlin/experiments.csv,10
8,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,/home/max/Documents/PhD/MAPF/LNS/MAPF_LNS_SAT/...,/home/max/Documents/PhD/MAPF/LNS/MAPF_LNS_SAT/...,10,8,42,1,375,1536.32,69,...,0.00,0.04,0.10,0.07,0.01,LNS,LNS,Berlin,../lns_clean/logs/LNS/Berlin/experiments.csv,10
11,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,/home/max/Documents/PhD/MAPF/LNS/MAPF_LNS_SAT/...,/home/max/Documents/PhD/MAPF/LNS/MAPF_LNS_SAT/...,10,11,42,1,388,1425.64,24,...,0.00,0.02,0.05,0.04,0.01,LNS,LNS,Berlin,../lns_clean/logs/LNS/Berlin/experiments.csv,10
13,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,/home/max/Documents/PhD/MAPF/LNS/MAPF_LNS_SAT/...,/home/max/Documents/PhD/MAPF/LNS/MAPF_LNS_SAT/...,10,13,42,1,443,1597.93,24,...,0.00,0.01,0.05,0.04,0.01,LNS,LNS,Berlin,../lns_clean/logs/LNS/Berlin/experiments.csv,10


In [187]:
def summarize_metric(group: pd.DataFrame, column: str) -> pd.Series:
    """Compute basic stats plus total for a CNF metric column."""

    stats = group[column].agg(["mean", "median", "std", "min", "max", "sum"])
    stats.rename({"sum": "total"}, inplace=True)
    stats["min_experiment_id"] = group.loc[group[column].idxmin(), "experiment_id"]
    stats["max_experiment_id"] = group.loc[group[column].idxmax(), "experiment_id"]
    # same experiment count for both metrics; attach it once later
    return stats


def summarize_cnf(experiments: pd.DataFrame) -> pd.DataFrame:
    summaries = []
    for map_category, group in experiments.groupby("map_category", sort=True):
        clauses = summarize_metric(group, "total_cnf_clauses").add_prefix("clauses_")
        variables = summarize_metric(group, "total_cnf_variables").add_prefix("variables_")
        combined = pd.concat([clauses, variables])
        combined["num_experiments"] = len(group)
        combined.name = map_category
        summaries.append(combined)

    summary_df = pd.DataFrame(summaries)
    column_order = [
        "num_experiments",
        "clauses_mean",
        "clauses_median",
        "clauses_std",
        "clauses_min",
        "clauses_min_experiment_id",
        "clauses_max",
        "clauses_max_experiment_id",
        "clauses_total",
        "variables_mean",
        "variables_median",
        "variables_std",
        "variables_min",
        "variables_min_experiment_id",
        "variables_max",
        "variables_max_experiment_id",
        "variables_total",
    ]
    return summary_df[column_order]


In [188]:
# Analyze zero-CNF experiments before dropping them

def summarize_zero_cnf(zero_df: pd.DataFrame, scenario_label: str):
    """Print compact stats about zero-CNF experiments for one scenario.

    Uses the concrete map file (from `map_path`) so different maps in the
    same folder (e.g., multiple Warehouse2-2 layouts) are treated separately.

    Shows:
    - total number of zero-CNF experiments
    - per-map-file counts
    - per (map-file, num_agents) counts
    """

    total = len(zero_df)
    print(f"{scenario_label}: total zero-CNF experiments = {total}")
    if total == 0:
        return

    zero_df = zero_df.copy()

    # Derive a concrete map identifier: file name from map_path when available
    if "map_path" in zero_df.columns:
        zero_df["map_file"] = zero_df["map_path"].apply(lambda p: Path(str(p)).name)
        map_col = "map_file"
    else:
        map_col = "map_name"

    # How many zero experiments on each concrete map?
    by_map = (
        zero_df.groupby(map_col, dropna=False)
        .size()
        .reset_index(name="zero_cnf_runs")
        .rename(columns={map_col: "map"})
    )
    print(f"{scenario_label}: zero-CNF runs per map file")
    display(by_map.sort_values("map"))

    # For maps that had any zero-CNF experiments, how many were at each agent count?
    if "num_agents" in zero_df.columns:
        by_map_agents = (
            zero_df.groupby([map_col, "num_agents"], dropna=False)
            .size()
            .reset_index(name="zero_cnf_runs")
            .rename(columns={map_col: "map"})
        )
        print(f"{scenario_label}: zero-CNF runs per (map file, num_agents)")
        display(by_map_agents.sort_values(["map", "num_agents"]))
    else:
        print(f"{scenario_label}: column 'num_agents' not found; skipping per-agent breakdown.")


# Run the zero-CNF analysis for each scenario BEFORE filtering them out
summarize_zero_cnf(lns_zero_cnf, "LNS")
summarize_zero_cnf(whole_zero_cnf, "Whole solve")


LNS: total zero-CNF experiments = 411
LNS: zero-CNF runs per map file


,map,zero_cnf_runs
0,Berlin_1_256.map,91
1,Paris_1_256.map,90
2,empty-16-16.map,7
3,empty-32-32.map,19
4,empty-48-48.map,44
5,room-32-32-4.map,2
6,room-64-64-16.map,6
7,room-64-64-8.map,4
8,warehouse-10-20-10-2-2.map,54
9,warehouse-20-40-10-2-2.map,94


LNS: zero-CNF runs per (map file, num_agents)


,map,num_agents,zero_cnf_runs
0,Berlin_1_256.map,10,66
1,Berlin_1_256.map,20,25
2,Paris_1_256.map,10,66
3,Paris_1_256.map,20,24
4,empty-16-16.map,10,7
5,empty-32-32.map,10,19
6,empty-48-48.map,10,42
7,empty-48-48.map,20,2
8,room-32-32-4.map,10,2
9,room-64-64-16.map,10,6


Whole solve: total zero-CNF experiments = 28
Whole solve: zero-CNF runs per map file


,map,zero_cnf_runs
0,Berlin_1_256.map,3
1,Paris_1_256.map,1
2,empty-16-16.map,7
3,empty-32-32.map,5
4,room-32-32-4.map,2
5,room-64-64-8.map,1
6,warehouse-10-20-10-2-2.map,6
7,warehouse-20-40-10-2-2.map,3


Whole solve: zero-CNF runs per (map file, num_agents)


,map,num_agents,zero_cnf_runs
0,Berlin_1_256.map,10,3
1,Paris_1_256.map,10,1
2,empty-16-16.map,10,7
3,empty-32-32.map,10,5
4,room-32-32-4.map,10,2
5,room-64-64-8.map,10,1
6,warehouse-10-20-10-2-2.map,10,6
7,warehouse-20-40-10-2-2.map,10,3


In [189]:
lns_summary = summarize_cnf(lns_experiments)
whole_summary = summarize_cnf(whole_experiments)

lns_summary.style.format(precision=2)


,num_experiments,clauses_mean,clauses_median,clauses_std,clauses_min,clauses_min_experiment_id,clauses_max,clauses_max_experiment_id,clauses_total,variables_mean,variables_median,variables_std,variables_min,variables_min_experiment_id,variables_max,variables_max_experiment_id,variables_total
LNS,2213,148845.01,686.00,1616980.50,12.00,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_mapf-map_warehouse-10-20-10-2-2_map-_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_mapf-scen-even_scen-even_warehouse-10-20-10-2-2-even-2_scen-agents10-idx37-seed42-1761041218903,43401293.00,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_mapf-map_room-32-32-4_map-_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_mapf-scen-even_scen-even_room-32-32-4-even-7_scen-agents20-idx1-seed42-1760992278564,329394005.00,4754.75,275.00,31914.04,7.00,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_mapf-map_Berlin_1_256_map-_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_mapf-scen-even_scen-even_Berlin_1_256-even-1_scen-agents10-idx33-seed42-1761056871874,676537.00,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_mapf-map_room-32-32-4_map-_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_mapf-scen-even_scen-even_room-32-32-4-even-7_scen-agents20-idx1-seed42-1760992278564,10522255.00


In [190]:
# Add an "ALL" summary row with totals across all agent counts
summary_row = pd.Series({
    "num_runs": lns_zones_by_agents["num_runs"].sum(),
    "mean": lns_attempts_by_run["zones_attempted_total"].mean(),
    "median": lns_attempts_by_run["zones_attempted_total"].median(),
    "std": lns_attempts_by_run["zones_attempted_total"].std(),
    "min": lns_attempts_by_run["zones_attempted_total"].min(),
    "max": lns_attempts_by_run["zones_attempted_total"].max(),
    "total": lns_zones_by_agents["total"].sum(),
}, name="ALL")

lns_zones_by_agents_with_total = pd.concat([lns_zones_by_agents, summary_row.to_frame().T], axis=0)

print("LNS: zones attempted per run grouped by agent count (with ALL summary row)")
display(lns_zones_by_agents_with_total.round(2))


LNS: zones attempted per run grouped by agent count (with ALL summary row)


,num_runs,mean,median,std,min,max,total
10,590.00,3.27,2.00,4.18,1.00,57.00,1932.00
20,673.00,5.60,4.00,4.94,1.00,32.00,3766.00
50,566.00,22.19,22.00,11.00,2.00,84.00,12561.00
100,316.00,81.61,89.00,25.00,4.00,141.00,25790.00
200,68.00,272.22,228.00,94.56,140.00,438.00,18511.00
ALL,2213.00,28.27,8.00,54.51,1.00,438.00,62560.00


In [191]:
# LNS solved zones analysis by agent count
# Count unique solved zones (not expansion steps)

def summarize_solved_zones_by_agents(zones_df: pd.DataFrame) -> pd.DataFrame:
    """Summarize total unique solved zones per run, grouped by agent_count.
    
    Only counts unique zones (identified by original boundaries), not expansion steps.
    Only includes zones that were successfully solved.
    """
    
    grouped = zones_df.groupby("agent_count")
    
    def analyze_solved_zones(group):
        """Count unique solved zones per experiment."""
        # Get unique zones for this agent count group
        unique_zones = identify_unique_zones(group)
        
        # For each experiment, count how many zones were solved
        # Group zones by experiment_id to count per run
        solved_zones_per_run = {}
        
        for zone_group in unique_zones:
            # Find final state (highest zone_attempt)
            max_attempt_idx = zone_group["zone_attempt"].idxmax()
            final_state = zone_group.loc[max_attempt_idx]
            
            # Only count if solved
            if final_state["solved"] == 1:
                exp_id = final_state["experiment_id"]
                if exp_id not in solved_zones_per_run:
                    solved_zones_per_run[exp_id] = 0
                solved_zones_per_run[exp_id] += 1
        
        # Convert to DataFrame for analysis
        if solved_zones_per_run:
            runs_df = pd.DataFrame({
                "experiment_id": list(solved_zones_per_run.keys()),
                "solved_zones_count": list(solved_zones_per_run.values())
            })
            
            return pd.Series({
                "num_runs": len(runs_df),
                "mean": runs_df["solved_zones_count"].mean(),
                "median": runs_df["solved_zones_count"].median(),
                "std": runs_df["solved_zones_count"].std(),
                "min": runs_df["solved_zones_count"].min(),
                "max": runs_df["solved_zones_count"].max(),
                "total": runs_df["solved_zones_count"].sum(),
            })
        else:
            return pd.Series({
                "num_runs": 0,
                "mean": 0,
                "median": 0,
                "std": 0,
                "min": 0,
                "max": 0,
                "total": 0,
            })
    
    stats = grouped.apply(analyze_solved_zones)
    
    # Reorder columns for readability
    stats = stats[["num_runs", "mean", "median", "std", "min", "max", "total"]]
    return stats.sort_index()


# Analyze solved zones (requires lns_local_zones to be loaded)
try:
    _ = lns_local_zones
except NameError:
    print("Error: lns_local_zones not loaded. Please run the local zones loading cell first.")
else:
    lns_solved_zones_by_agents = summarize_solved_zones_by_agents(lns_local_zones)
    
    # Add an "ALL" summary row with totals across all agent counts
    # We need to recalculate for all agent counts together
    all_unique_zones = identify_unique_zones(lns_local_zones)
    solved_zones_per_run_all = {}
    
    for zone_group in all_unique_zones:
        max_attempt_idx = zone_group["zone_attempt"].idxmax()
        final_state = zone_group.loc[max_attempt_idx]
        
        if final_state["solved"] == 1:
            exp_id = final_state["experiment_id"]
            if exp_id not in solved_zones_per_run_all:
                solved_zones_per_run_all[exp_id] = 0
            solved_zones_per_run_all[exp_id] += 1
    
    if solved_zones_per_run_all:
        all_runs_df = pd.DataFrame({
            "experiment_id": list(solved_zones_per_run_all.keys()),
            "solved_zones_count": list(solved_zones_per_run_all.values())
        })
        
        summary_row = pd.Series({
            "num_runs": len(all_runs_df),
            "mean": all_runs_df["solved_zones_count"].mean(),
            "median": all_runs_df["solved_zones_count"].median(),
            "std": all_runs_df["solved_zones_count"].std(),
            "min": all_runs_df["solved_zones_count"].min(),
            "max": all_runs_df["solved_zones_count"].max(),
            "total": all_runs_df["solved_zones_count"].sum(),
        }, name="ALL")
    else:
        summary_row = pd.Series({
            "num_runs": 0,
            "mean": 0,
            "median": 0,
            "std": 0,
            "min": 0,
            "max": 0,
            "total": 0,
        }, name="ALL")
    
    lns_solved_zones_with_total = pd.concat([lns_solved_zones_by_agents, summary_row.to_frame().T], axis=0)
    
    print("LNS: unique solved zones per run grouped by agent count (with ALL summary row)")
    print("Note: Only counts unique zones (not expansion steps), only includes successfully solved zones")
    display(lns_solved_zones_with_total.round(2))


/tmp/ipykernel_720/1887456185.py:61: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  stats = grouped.apply(analyze_solved_zones)


LNS: unique solved zones per run grouped by agent count (with ALL summary row)
Note: Only counts unique zones (not expansion steps), only includes successfully solved zones


,num_runs,mean,median,std,min,max,total
10,590.00,2.37,2.00,1.72,1.00,15.00,1397.00
20,673.00,4.58,4.00,3.45,1.00,21.00,3085.00
50,566.00,18.76,18.00,9.25,1.00,63.00,10619.00
100,316.00,68.77,75.50,20.97,1.00,115.00,21730.00
200,68.00,223.74,186.50,74.04,121.00,353.00,15214.00
ALL,2213.00,23.52,6.00,44.88,1.00,353.00,52045.00


In [192]:
# Filter makespan attempts to only include concluded experiments
# Load concluded experiment IDs from experiments.csv
lns_root = SCENARIO_FOLDERS["LNS"]
experiment_csv_paths = list(lns_root.rglob("experiments.csv"))
if experiment_csv_paths:
    concluded_experiment_ids = set()
    for exp_csv_path in experiment_csv_paths:
        exp_raw = pd.read_csv(exp_csv_path)
        exp_df = normalize_columns(exp_raw)
        if "experiment_id" in exp_df.columns:
            concluded_experiment_ids.update(exp_df["experiment_id"].unique())
    
    total_before = len(lns_attempts)
    lns_attempts = lns_attempts[lns_attempts["experiment_id"].isin(concluded_experiment_ids)]
    print(f"Filtered makespan attempts: {total_before} total rows -> {len(lns_attempts)} rows from concluded experiments")
    print(f"Excluded {total_before - len(lns_attempts)} attempt rows from unfinished experiments")


Filtered makespan attempts: 2627 total rows -> 2627 rows from concluded experiments
Excluded 0 attempt rows from unfinished experiments


In [193]:
whole_summary.style.format(precision=2)


,num_experiments,clauses_mean,clauses_median,clauses_std,clauses_min,clauses_min_experiment_id,clauses_max,clauses_max_experiment_id,clauses_total,variables_mean,variables_median,variables_std,variables_min,variables_min_experiment_id,variables_max,variables_max_experiment_id,variables_total
WholeSolve,278,1916342.74,709930.00,2672259.02,18326.00,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_mapf-map_empty-16-16_map-_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_mapf-scen-even_scen-even_empty-16-16-even-8_scen-agents10-idx8-seed42-1760967891334,13276991.00,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_mapf-map_empty-32-32_map-_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_mapf-scen-even_scen-even_empty-32-32-even-1_scen-agents10-idx12-seed42-1760969799710,532743283.00,26706.88,17002.00,25602.53,1626.00,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_mapf-map_empty-16-16_map-_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_mapf-scen-even_scen-even_empty-16-16-even-8_scen-agents10-idx8-seed42-1760967891334,155613.00,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_mapf-map_empty-16-16_map-_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_mapf-scen-even_scen-even_empty-16-16-even-1_scen-agents100-idx0-seed42-1760968889743,7424512.00


In [194]:
# LNS zones analysis by agent count

LNS_REQUIRED_MAKESPAN_COLUMNS = {"experiment_id", "makespan_attempt", "zones_attempted"}


def load_lns_makespan_attempts() -> pd.DataFrame:
    """Load all LNS makespan_attempts.csv files into a single DataFrame."""

    lns_root = SCENARIO_FOLDERS["LNS"]
    csv_paths = list(lns_root.rglob("makespan_attempts.csv"))
    if not csv_paths:
        raise FileNotFoundError(f"No makespan_attempts.csv files found under {lns_root}.")

    frames: list[pd.DataFrame] = []
    for csv_path in csv_paths:
        raw = pd.read_csv(csv_path)
        df = normalize_columns(raw)

        missing = LNS_REQUIRED_MAKESPAN_COLUMNS - set(df.columns)
        if missing:
            raise KeyError(
                f"{csv_path} is missing required columns for makespan analysis: {', '.join(sorted(missing))}"
            )

        # Attach agent count per row using the shared experiment_id encoding
        df["agent_count"] = df["experiment_id"].apply(extract_agent_count)
        frames.append(df)

    return pd.concat(frames, ignore_index=True)


lns_attempts = load_lns_makespan_attempts()

# Collapse multiple makespan_attempt rows into a single run per experiment
lns_attempts_by_run = (
    lns_attempts
    .groupby(["experiment_id", "agent_count"], as_index=False)
    .agg(zones_attempted_total=("zones_attempted", "sum"))
)

# Exclude runs with instant solutions (no zones built at all)
instant_runs = lns_attempts_by_run[lns_attempts_by_run["zones_attempted_total"] == 0]
print("Runs with instant solutions (no zones built):", len(instant_runs))
if not instant_runs.empty:
    print("Instant-solution runs by agent count:")
    display(
        instant_runs.groupby("agent_count").size().reset_index(name="instant_runs")
    )

lns_attempts_by_run = lns_attempts_by_run[lns_attempts_by_run["zones_attempted_total"] > 0]


def summarize_zones_by_agents(runs_df: pd.DataFrame) -> pd.DataFrame:
    """Summarize total zones attempted per run, grouped by agent_count."""

    grouped = runs_df.groupby("agent_count")
    stats = grouped["zones_attempted_total"].agg(
        mean="mean",
        median="median",
        std="std",
        min="min",
        max="max",
        total="sum",
    )
    stats["num_runs"] = grouped.size()

    # Reorder columns for readability
    stats = stats[["num_runs", "mean", "median", "std", "min", "max", "total"]]
    return stats.sort_index()


lns_zones_by_agents = summarize_zones_by_agents(lns_attempts_by_run)

print("LNS: zones attempted per run grouped by agent count")
lns_zones_by_agents.round(2)


Runs with instant solutions (no zones built): 411
Instant-solution runs by agent count:


,agent_count,instant_runs
0,10,333
1,20,78


LNS: zones attempted per run grouped by agent count


,num_runs,mean,median,std,min,max,total
agent_count,,,,,,,
10,590,3.27,2.00,4.18,1,57,1932
20,673,5.60,4.00,4.94,1,32,3766
50,566,22.19,22.00,11.00,2,84,12561
100,316,81.61,89.00,25.00,4,141,25790
200,68,272.22,228.00,94.56,140,438,18511


## Optional: summaries by agent count

The snippet below groups by map category and agent count, which can be helpful if you want to compare
how CNF growth scales with the number of agents.


In [195]:
def summarize_by_agent(experiments: pd.DataFrame) -> pd.DataFrame:
    grouped = experiments.groupby(["map_category", "agent_count"], sort=True)
    return grouped.agg(
        clauses_mean=("total_cnf_clauses", "mean"),
        clauses_std=("total_cnf_clauses", "std"),
        variables_mean=("total_cnf_variables", "mean"),
        variables_std=("total_cnf_variables", "std"),
        runs=("experiment_id", "count"),
    ).reset_index()


# Optionally exclude LNS runs that never produced a CNF (0 clauses and 0 variables)
# Requires the `lns_zero_cnf` DataFrame from the earlier CNF filtering cell.
if "lns_zero_cnf" in globals():
    zero_cnf_ids = set(lns_zero_cnf["experiment_id"])
    lns_experiments_filtered = lns_experiments[~lns_experiments["experiment_id"].isin(zero_cnf_ids)]

    print(
        "LNS: excluding",
        len(lns_experiments) - len(lns_experiments_filtered),
        "experiments with CNF size 0 (no clauses and no variables) from CNF agent summary",
    )

    if len(zero_cnf_ids) > 0:
        print("LNS experiments with CNF size 0 by agent count:")
        display(
            lns_experiments[lns_experiments["experiment_id"].isin(zero_cnf_ids)]
            .groupby("agent_count")["experiment_id"]
            .nunique()
            .reset_index(name="zero_cnf_experiments")
        )
else:
    lns_experiments_filtered = lns_experiments


data_by_agent = {
    "LNS": summarize_by_agent(lns_experiments_filtered),
    "Whole solve": summarize_by_agent(whole_experiments),
}

for scenario_name, agent_summary in data_by_agent.items():
    display(f"Agent-level summary for {scenario_name}")
    display(agent_summary.sort_values(["map_category", "agent_count"]).round(2))


LNS: excluding 0 experiments with CNF size 0 (no clauses and no variables) from CNF agent summary
LNS experiments with CNF size 0 by agent count:


,agent_count,zero_cnf_experiments


'Agent-level summary for LNS'

,map_category,agent_count,clauses_mean,clauses_std,variables_mean,variables_std,runs
0,LNS,10,174667.14,1765876.74,3005.02,27574.89,590
1,LNS,20,150732.61,2161566.07,3000.93,34692.93,673
2,LNS,50,171736.12,1123107.09,7438.83,40281.55,566
3,LNS,100,40912.84,464745.32,3839.41,9223.63,316
4,LNS,200,217150.71,568180.99,19206.31,21253.61,68


'Agent-level summary for Whole solve'

,map_category,agent_count,clauses_mean,clauses_std,variables_mean,variables_std,runs
0,WholeSolve,10,1986700.33,2746490.53,21572.68,21037.87,151
1,WholeSolve,20,1520819.28,2592914.61,25500.83,22364.20,109
2,WholeSolve,50,3456129.24,1186318.08,72460.82,17358.16,17
3,WholeSolve,100,8228035.00,NaN,155613.00,NaN,1


## LNS Local Zones Analysis

This section analyzes the detailed zone-level data from `local_zones.csv` files. Each row represents a zone that was attempted during a makespan attempt.

**Key metrics:**
- **Zone positions**: Number of positions covered by the zone
- **Conflicts**: Number of conflicts within the zone
- **Zone attempt**: 0 indicates a new zone, >1 indicates the zone was extended (expanded) after initial attempts
- **Waiting attempts**: If >1, waiting time was used to try solving the zone before expanding it
- **Solved**: 1 if the zone was successfully solved, 0 otherwise

**Analysis includes:**
1. Statistics on conflicts and zone positions per agent count (mean, median, min, max)
2. Zone building statistics per agent count:
   - Total zones built
   - Zones that were extended (zone_attempt > 1)
   - Zones solved using waiting time (waiting_attempts > 1 AND solved = 1)


In [196]:
# Find and display the largest zone
if not lns_local_zones.empty:
    largest_zone_idx = lns_local_zones["zone_positions"].idxmax()
    largest_zone = lns_local_zones.loc[largest_zone_idx]
    
    # Extract map name from experiment_id or csv_path if available
    map_name = "Unknown"
    if "map_name" in lns_local_zones.columns:
        map_name = largest_zone["map_name"]
    elif "csv_path" in lns_local_zones.columns:
        # Try to extract from path
        csv_path = largest_zone.get("csv_path", "")
        if csv_path:
            from pathlib import Path
            map_name = Path(csv_path).parent.name
    
    print("=" * 80)
    print("LARGEST ZONE FOUND")
    print("=" * 80)
    print(f"Map: {map_name}")
    print(f"Experiment ID: {largest_zone['experiment_id']}")
    print(f"Makespan Attempt: {largest_zone.get('makespan_attempt', 'N/A')}")
    print(f"Zone Attempt: {largest_zone.get('zone_attempt', 'N/A')}")
    print(f"Time Window: {largest_zone.get('start_t', 'N/A')} to {largest_zone.get('end_t', 'N/A')}")
    print(f"Zone Positions (Size): {largest_zone['zone_positions']}")
    print(f"Conflicts: {largest_zone.get('conflicts', 'N/A')}")
    print(f"Agents: {largest_zone.get('agents', 'N/A')}")
    print(f"Agent Count: {largest_zone.get('agent_count', 'N/A')}")
    print(f"Zone Fraction: {largest_zone.get('zone_fraction', 'N/A'):.6f}" if 'zone_fraction' in largest_zone else "")
    print("=" * 80)
    
    # Also create a nice DataFrame display
    largest_zone_display = pd.DataFrame({
        "Metric": [
            "Map",
            "Zone Positions (Size)",
            "Conflicts",
            "Agents",
            "Agent Count",
            "Time Window",
            "Makespan Attempt",
            "Zone Attempt",
            "Zone Fraction",
        ],
        "Value": [
            map_name,
            largest_zone["zone_positions"],
            largest_zone.get("conflicts", "N/A"),
            largest_zone.get("agents", "N/A"),
            largest_zone.get("agent_count", "N/A"),
            f"{largest_zone.get('start_t', 'N/A')} - {largest_zone.get('end_t', 'N/A')}",
            largest_zone.get("makespan_attempt", "N/A"),
            largest_zone.get("zone_attempt", "N/A"),
            f"{largest_zone.get('zone_fraction', 0):.6f}" if 'zone_fraction' in largest_zone else "N/A",
        ]
    })
    display(largest_zone_display)
    
    # Find and display solved zones with 4 and 5 positions
    print("\n")
    print("=" * 80)
    print("SMALLEST ZONES: Solved zones with 4 and 5 positions")
    print("=" * 80)
    
    # Helper function to separate instant solve and waiting time solve
    def categorize_solved_zones(zones_df):
        """Categorize solved zones into instant solve and waiting time solve."""
        # Instant solve: solved == 1, zone_attempt == 0, waiting_attempts == 1
        instant_solve = zones_df[
            (zones_df["zone_attempt"] == 0) &
            (zones_df["waiting_attempts"] == 1)
        ]
        # Waiting time solve: solved == 1, waiting_attempts > 1
        waiting_solve = zones_df[
            zones_df["waiting_attempts"] > 1
        ]
        return instant_solve, waiting_solve
    
    # Filter for solved zones with 4 positions
    solved_4pos = lns_local_zones[
        (lns_local_zones["zone_positions"] == 4) & 
        (lns_local_zones["solved"] == 1)
    ]
    instant_4pos, waiting_4pos = categorize_solved_zones(solved_4pos)
    
    # Filter for solved zones with 5 positions
    solved_5pos = lns_local_zones[
        (lns_local_zones["zone_positions"] == 5) & 
        (lns_local_zones["solved"] == 1)
    ]
    instant_5pos, waiting_5pos = categorize_solved_zones(solved_5pos)
    
    # Combined totals for 4+5 positions
    total_4_5_instant = len(instant_4pos) + len(instant_5pos)
    total_4_5_waiting = len(waiting_4pos) + len(waiting_5pos)
    total_4_5 = len(solved_4pos) + len(solved_5pos)
    
    # Print summary
    print(f"\n4-POSITION ZONES:")
    print(f"  Total solved: {len(solved_4pos)}")
    print(f"  Instant solve: {len(instant_4pos)} ({len(instant_4pos)/len(solved_4pos)*100:.2f}%)" if len(solved_4pos) > 0 else "  Instant solve: 0")
    print(f"  Waiting time solve: {len(waiting_4pos)} ({len(waiting_4pos)/len(solved_4pos)*100:.2f}%)" if len(solved_4pos) > 0 else "  Waiting time solve: 0")
    
    print(f"\n5-POSITION ZONES:")
    print(f"  Total solved: {len(solved_5pos)}")
    print(f"  Instant solve: {len(instant_5pos)} ({len(instant_5pos)/len(solved_5pos)*100:.2f}%)" if len(solved_5pos) > 0 else "  Instant solve: 0")
    print(f"  Waiting time solve: {len(waiting_5pos)} ({len(waiting_5pos)/len(solved_5pos)*100:.2f}%)" if len(solved_5pos) > 0 else "  Waiting time solve: 0")
    
    print(f"\nCOMBINED (4 + 5 POSITIONS):")
    print(f"  Total solved: {total_4_5}")
    print(f"  Instant solve: {total_4_5_instant} ({total_4_5_instant/total_4_5*100:.2f}%)" if total_4_5 > 0 else "  Instant solve: 0")
    print(f"  Waiting time solve: {total_4_5_waiting} ({total_4_5_waiting/total_4_5*100:.2f}%)" if total_4_5 > 0 else "  Waiting time solve: 0")
    
    print("=" * 80)
    
    # Create summary DataFrame
    summary_data = {
        "Category": [
            "4 positions - Total solved",
            "4 positions - Instant solve",
            "4 positions - Instant solve %",
            "4 positions - Waiting time solve",
            "4 positions - Waiting time solve %",
            "",
            "5 positions - Total solved",
            "5 positions - Instant solve",
            "5 positions - Instant solve %",
            "5 positions - Waiting time solve",
            "5 positions - Waiting time solve %",
            "",
            "Combined (4+5) - Total solved",
            "Combined (4+5) - Instant solve",
            "Combined (4+5) - Instant solve %",
            "Combined (4+5) - Waiting time solve",
            "Combined (4+5) - Waiting time solve %",
        ],
        "Count": [
            len(solved_4pos),
            len(instant_4pos),
            f"{len(instant_4pos)/len(solved_4pos)*100:.2f}%" if len(solved_4pos) > 0 else "0.00%",
            len(waiting_4pos),
            f"{len(waiting_4pos)/len(solved_4pos)*100:.2f}%" if len(solved_4pos) > 0 else "0.00%",
            "",
            len(solved_5pos),
            len(instant_5pos),
            f"{len(instant_5pos)/len(solved_5pos)*100:.2f}%" if len(solved_5pos) > 0 else "0.00%",
            len(waiting_5pos),
            f"{len(waiting_5pos)/len(solved_5pos)*100:.2f}%" if len(solved_5pos) > 0 else "0.00%",
            "",
            total_4_5,
            total_4_5_instant,
            f"{total_4_5_instant/total_4_5*100:.2f}%" if total_4_5 > 0 else "0.00%",
            total_4_5_waiting,
            f"{total_4_5_waiting/total_4_5*100:.2f}%" if total_4_5 > 0 else "0.00%",
        ]
    }
    
    smallest_zone_summary = pd.DataFrame(summary_data)
    display(smallest_zone_summary)
else:
    print("No local zones data available")


LARGEST ZONE FOUND
Map: Unknown
Experiment ID: _home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_mapf-map_room-32-32-4_map-_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_mapf-scen-even_scen-even_room-32-32-4-even-3_scen-agents10-idx12-seed42-1760991891204
Makespan Attempt: 0
Zone Attempt: 36
Time Window: 0 to 44
Zone Positions (Size): 682
Conflicts: 1
Agents: 10
Agent Count: 10
Zone Fraction: 1.000000


,Metric,Value
0,Map,Unknown
1,Zone Positions (Size),682
2,Conflicts,1
3,Agents,10
4,Agent Count,10
5,Time Window,0 - 44
6,Makespan Attempt,0
7,Zone Attempt,36
8,Zone Fraction,1.000000




SMALLEST ZONES: Solved zones with 4 and 5 positions

4-POSITION ZONES:
  Total solved: 8776
  Instant solve: 8211 (93.56%)
  Waiting time solve: 565 (6.44%)

5-POSITION ZONES:
  Total solved: 20023
  Instant solve: 14088 (70.36%)
  Waiting time solve: 5935 (29.64%)

COMBINED (4 + 5 POSITIONS):
  Total solved: 28799
  Instant solve: 22299 (77.43%)
  Waiting time solve: 6500 (22.57%)


,Category,Count
0,4 positions - Total solved,8776
1,4 positions - Instant solve,8211
2,4 positions - Instant solve %,93.56%
3,4 positions - Waiting time solve,565
4,4 positions - Waiting time solve %,6.44%
5,,
6,5 positions - Total solved,20023
7,5 positions - Instant solve,14088
8,5 positions - Instant solve %,70.36%
9,5 positions - Waiting time solve,5935


In [197]:
# LNS Local Zones Analysis

def load_lns_local_zones() -> pd.DataFrame:
    """Load all LNS local_zones.csv files into a single DataFrame.
    
    Only includes zones from concluded experiments (those present in experiments.csv).
    Filters out zones from unfinished experiments.
    """
    
    # First, load all concluded experiment IDs from experiments.csv files
    lns_root = SCENARIO_FOLDERS["LNS"]
    experiment_csv_paths = list(lns_root.rglob("experiments.csv"))
    if not experiment_csv_paths:
        raise FileNotFoundError(f"No experiments.csv files found under {lns_root}.")
    
    concluded_experiment_ids = set()
    for exp_csv_path in experiment_csv_paths:
        exp_raw = pd.read_csv(exp_csv_path)
        exp_df = normalize_columns(exp_raw)
        if "experiment_id" in exp_df.columns:
            concluded_experiment_ids.update(exp_df["experiment_id"].unique())
    
    print(f"Found {len(concluded_experiment_ids)} concluded experiments in experiments.csv files")
    
    # Now load local_zones.csv files and filter by concluded experiment IDs
    csv_paths = list(lns_root.rglob("local_zones.csv"))
    if not csv_paths:
        raise FileNotFoundError(f"No local_zones.csv files found under {lns_root}.")
    
    frames: list[pd.DataFrame] = []
    total_zones_before_filter = 0
    for csv_path in csv_paths:
        raw = pd.read_csv(csv_path)
        df = normalize_columns(raw)
        
        total_zones_before_filter += len(df)
        
        # Filter to only include zones from concluded experiments
        df = df[df["experiment_id"].isin(concluded_experiment_ids)]
        
        # Attach agent count per row using the shared experiment_id encoding
        df["agent_count"] = df["experiment_id"].apply(extract_agent_count)
        frames.append(df)
    
    result = pd.concat(frames, ignore_index=True)
    print(f"Filtered local zones: {total_zones_before_filter} total rows -> {len(result)} rows from concluded experiments")
    print(f"Excluded {total_zones_before_filter - len(result)} zone rows from unfinished experiments")
    
    return result


lns_local_zones = load_lns_local_zones()

# Statistics on conflicts and zone positions per agent count
def summarize_zone_metrics(zones_df: pd.DataFrame) -> pd.DataFrame:
    """Compute statistics on conflicts and zone_positions grouped by agent_count."""
    
    grouped = zones_df.groupby("agent_count")
    
    stats = pd.DataFrame({
        "conflicts_mean": grouped["conflicts"].mean(),
        "conflicts_median": grouped["conflicts"].median(),
        "conflicts_std": grouped["conflicts"].std(),
        "conflicts_min": grouped["conflicts"].min(),
        "conflicts_max": grouped["conflicts"].max(),
        "positions_mean": grouped["zone_positions"].mean(),
        "positions_median": grouped["zone_positions"].median(),
        "positions_std": grouped["zone_positions"].std(),
        "positions_min": grouped["zone_positions"].min(),
        "positions_max": grouped["zone_positions"].max(),
    })
    
    return stats.sort_index()


zone_metrics = summarize_zone_metrics(lns_local_zones)

print("LNS: Zone metrics (conflicts and positions) per agent count")
display(zone_metrics.round(2))

# Add an "ALL" summary row with aggregated statistics across all agent counts
summary_row = pd.Series({
    "conflicts_mean": lns_local_zones["conflicts"].mean(),
    "conflicts_median": lns_local_zones["conflicts"].median(),
    "conflicts_std": lns_local_zones["conflicts"].std(),
    "conflicts_min": lns_local_zones["conflicts"].min(),
    "conflicts_max": lns_local_zones["conflicts"].max(),
    "positions_mean": lns_local_zones["zone_positions"].mean(),
    "positions_median": lns_local_zones["zone_positions"].median(),
    "positions_std": lns_local_zones["zone_positions"].std(),
    "positions_min": lns_local_zones["zone_positions"].min(),
    "positions_max": lns_local_zones["zone_positions"].max(),
}, name="ALL")

zone_metrics_with_total = pd.concat([zone_metrics, summary_row.to_frame().T], axis=0)

print("\nLNS: Zone metrics (conflicts and positions) per agent count (with ALL summary row)")
display(zone_metrics_with_total.round(2))

# Zone building statistics per agent count

def identify_unique_zones(zones_df: pd.DataFrame):
    """Identify unique zones by their original boundaries (zone_attempt=0).
    
    Returns a list of DataFrames, each containing all rows for one unique zone.
    Zones are identified by their original boundaries from zone_attempt=0.
    Expansion rows (zone_attempt > 0) are matched to their original zone by sequence.
    """
    # Group by experiment_id and makespan_attempt first
    exp_makespan_groups = zones_df.groupby(["experiment_id", "makespan_attempt"])
    zone_groups_list = []
    
    for (exp_id, makespan), exp_group in exp_makespan_groups:
        # Preserve original order (rows are already in sequence)
        # Find all zone_attempt=0 rows (original zone definitions) in order
        original_mask = exp_group["zone_attempt"] == 0
        original_indices = exp_group[original_mask].index.tolist()
        
        # For each original zone, collect all its rows (including expansions)
        for i, orig_idx in enumerate(original_indices):
            orig_row = exp_group.loc[orig_idx]
            orig_start = orig_row["start_t"]
            orig_end = orig_row["end_t"]
            
            # Find the end index: either the next original zone or the end of the group
            if i + 1 < len(original_indices):
                end_idx = original_indices[i + 1]
                # Get all rows from orig_idx up to (but not including) end_idx
                zone_rows = exp_group.loc[orig_idx:end_idx].iloc[:-1] if end_idx > orig_idx else exp_group.loc[[orig_idx]]
            else:
                zone_rows = exp_group.loc[orig_idx:]
            
            zone_groups_list.append(zone_rows)
    
    return zone_groups_list


def summarize_zone_building(zones_df: pd.DataFrame) -> pd.DataFrame:
    """Compute zone building statistics grouped by agent_count.
    
    Counts unique zones (not rows). Each zone is identified by its
    original boundaries (from zone_attempt=0), not expanded boundaries.
    """
    
    grouped = zones_df.groupby("agent_count")
    
    def analyze_zones(group):
        """Analyze unique zones in a group and return counts."""
        # Use the helper function to identify unique zones
        zone_groups = identify_unique_zones(group)
        
        total_zones = len(zone_groups)
        zones_extended = 0
        zone_extensions = 0  # total number of extension attempts
        zones_solved_with_waiting = 0
        zones_solved_first_try = 0
        zones_extended_solved_without_waiting = 0  # extended and solved, but without waiting time
        zones_unsolved = 0
        
        for zone_group in zone_groups:
            # Check if zone was extended (has any row with zone_attempt > 0)
            was_extended = (zone_group["zone_attempt"] > 0).any()
            if was_extended:
                zones_extended += 1
                # Count extension attempts (rows with zone_attempt > 0)
                zone_extensions += (zone_group["zone_attempt"] > 0).sum()
            
            # Find the final state of the zone (highest zone_attempt)
            max_attempt_idx = zone_group["zone_attempt"].idxmax()
            final_state = zone_group.loc[max_attempt_idx]
            
            # Check final solved state
            if final_state["solved"] == 0:
                zones_unsolved += 1
            else:
                # Zone was solved - categorize how
                if not was_extended and final_state["waiting_attempts"] == 1:
                    # Solved on first try: no extension, no waiting
                    zones_solved_first_try += 1
                elif final_state["waiting_attempts"] > 1:
                    # Solved with waiting time (could be with or without extension)
                    zones_solved_with_waiting += 1
                else:
                    # Extended and solved, but without waiting time
                    zones_extended_solved_without_waiting += 1
        
        return pd.Series({
            "total_zones": total_zones,
            "zones_extended": zones_extended,
            "zone_extensions": zone_extensions,
            "zones_solved_with_waiting": zones_solved_with_waiting,
            "zones_solved_first_try": zones_solved_first_try,
            "zones_extended_solved_without_waiting": zones_extended_solved_without_waiting,
            "zones_unsolved": zones_unsolved,
        })
    
    stats = grouped.apply(analyze_zones)
    
    # Add percentages
    stats["pct_extended"] = (stats["zones_extended"] / stats["total_zones"] * 100).round(2)
    stats["pct_solved_with_waiting"] = (stats["zones_solved_with_waiting"] / stats["total_zones"] * 100).round(2)
    stats["pct_solved_first_try"] = (stats["zones_solved_first_try"] / stats["total_zones"] * 100).round(2)
    stats["pct_extended_solved_without_waiting"] = (stats["zones_extended_solved_without_waiting"] / stats["total_zones"] * 100).round(2)
    stats["pct_unsolved"] = (stats["zones_unsolved"] / stats["total_zones"] * 100).round(2)
    
    # Verification: check that solved categories + unsolved = total_zones
    stats["check_sum"] = (
        stats["zones_solved_first_try"] 
        + stats["zones_solved_with_waiting"] 
        + stats["zones_extended_solved_without_waiting"]
        + stats["zones_unsolved"]
    )
    
    return stats.sort_index()


zone_building_stats = summarize_zone_building(lns_local_zones)

# Add a summary row across all agent counts
summary_counts = zone_building_stats[[
    "total_zones",
    "zones_extended",
    "zone_extensions",
    "zones_solved_with_waiting",
    "zones_solved_first_try",
    "zones_extended_solved_without_waiting",
    "zones_unsolved",
]].sum()

summary_counts["pct_extended"] = (summary_counts["zones_extended"] / summary_counts["total_zones"] * 100).round(2)
summary_counts["pct_solved_with_waiting"] = (summary_counts["zones_solved_with_waiting"] / summary_counts["total_zones"] * 100).round(2)
summary_counts["pct_solved_first_try"] = (summary_counts["zones_solved_first_try"] / summary_counts["total_zones"] * 100).round(2)
summary_counts["pct_extended_solved_without_waiting"] = (summary_counts["zones_extended_solved_without_waiting"] / summary_counts["total_zones"] * 100).round(2)
summary_counts["pct_unsolved"] = (summary_counts["zones_unsolved"] / summary_counts["total_zones"] * 100).round(2)
summary_counts["check_sum"] = (
    summary_counts["zones_solved_first_try"]
    + summary_counts["zones_solved_with_waiting"]
    + summary_counts["zones_extended_solved_without_waiting"]
    + summary_counts["zones_unsolved"]
)

summary_row = summary_counts.to_frame().T
summary_row.index = ["ALL"]

zone_building_with_total = pd.concat([zone_building_stats, summary_row], axis=0)

print("\nLNS: Zone building statistics per agent count (with ALL summary row)")
display(zone_building_with_total)

# Detailed breakdown: How zones were solved
def summarize_zone_solving_methods(zones_df: pd.DataFrame) -> pd.DataFrame:
    """Break down zones by how they were solved (or not solved).
    
    Categories:
    - solved_no_expansion_no_waiting: solved on first attempt, no extension, no waiting
    - solved_no_expansion_with_waiting: solved without extension but used waiting time
    - solved_after_expansion_no_waiting: extended and solved without waiting (on first extension attempt)
    - solved_after_expansion_with_waiting: extended and solved using waiting time
    - unsolved: never solved
    """
    
    grouped = zones_df.groupby("agent_count")
    
    def analyze_solving_methods(group):
        # Use the helper function to identify unique zones by original boundaries
        zone_groups = identify_unique_zones(group)
        
        solved_no_expansion_no_waiting = 0
        solved_no_expansion_with_waiting = 0
        solved_after_expansion_no_waiting = 0
        solved_after_expansion_with_waiting = 0
        unsolved = 0
        
        for zone_group in zone_groups:
            was_extended = (zone_group["zone_attempt"] > 0).any()
            max_attempt_idx = zone_group["zone_attempt"].idxmax()
            final_state = zone_group.loc[max_attempt_idx]
            
            if final_state["solved"] == 0:
                unsolved += 1
            else:
                # Zone was solved
                if not was_extended:
                    # No expansion
                    if final_state["waiting_attempts"] == 1:
                        solved_no_expansion_no_waiting += 1
                    else:  # waiting_attempts > 1
                        solved_no_expansion_with_waiting += 1
                else:
                    # Was extended - check if solved on first extension attempt
                    min_extended_attempt = zone_group[zone_group["zone_attempt"] > 0]["zone_attempt"].min()
                    solved_on_first_extension = (final_state["zone_attempt"] == min_extended_attempt)
                    
                    if solved_on_first_extension and final_state["waiting_attempts"] == 1:
                        # Solved instantly after expansion (first extension attempt, no waiting)
                        solved_after_expansion_no_waiting += 1
                    elif final_state["waiting_attempts"] > 1:
                        # Solved with waiting time after expansion
                        solved_after_expansion_with_waiting += 1
                    else:
                        # Extended multiple times, solved without waiting (not on first extension)
                        solved_after_expansion_no_waiting += 1
        
        return pd.Series({
            "solved_no_expansion_no_waiting": solved_no_expansion_no_waiting,
            "solved_no_expansion_with_waiting": solved_no_expansion_with_waiting,
            "solved_after_expansion_no_waiting": solved_after_expansion_no_waiting,
            "solved_after_expansion_with_waiting": solved_after_expansion_with_waiting,
            "unsolved": unsolved,
        })
    
    stats = grouped.apply(analyze_solving_methods)
    
    # Calculate total and percentages
    stats["total_zones"] = (
        stats["solved_no_expansion_no_waiting"]
        + stats["solved_no_expansion_with_waiting"]
        + stats["solved_after_expansion_no_waiting"]
        + stats["solved_after_expansion_with_waiting"]
        + stats["unsolved"]
    )
    
    stats["pct_no_exp_no_wait"] = (stats["solved_no_expansion_no_waiting"] / stats["total_zones"] * 100).round(2)
    stats["pct_no_exp_with_wait"] = (stats["solved_no_expansion_with_waiting"] / stats["total_zones"] * 100).round(2)
    stats["pct_after_exp_no_wait"] = (stats["solved_after_expansion_no_waiting"] / stats["total_zones"] * 100).round(2)
    stats["pct_after_exp_with_wait"] = (stats["solved_after_expansion_with_waiting"] / stats["total_zones"] * 100).round(2)
    stats["pct_unsolved"] = (stats["unsolved"] / stats["total_zones"] * 100).round(2)
    
    # Reorder columns
    column_order = [
        "total_zones",
        "solved_no_expansion_no_waiting",
        "pct_no_exp_no_wait",
        "solved_no_expansion_with_waiting",
        "pct_no_exp_with_wait",
        "solved_after_expansion_no_waiting",
        "pct_after_exp_no_wait",
        "solved_after_expansion_with_waiting",
        "pct_after_exp_with_wait",
        "unsolved",
        "pct_unsolved",
    ]
    
    return stats[column_order].sort_index()


solving_methods_stats = summarize_zone_solving_methods(lns_local_zones)

# Add summary row
summary_solving = solving_methods_stats[[
    "solved_no_expansion_no_waiting",
    "solved_no_expansion_with_waiting",
    "solved_after_expansion_no_waiting",
    "solved_after_expansion_with_waiting",
    "unsolved",
]].sum()

summary_solving["total_zones"] = summary_solving.sum()
summary_solving["pct_no_exp_no_wait"] = (summary_solving["solved_no_expansion_no_waiting"] / summary_solving["total_zones"] * 100).round(2)
summary_solving["pct_no_exp_with_wait"] = (summary_solving["solved_no_expansion_with_waiting"] / summary_solving["total_zones"] * 100).round(2)
summary_solving["pct_after_exp_no_wait"] = (summary_solving["solved_after_expansion_no_waiting"] / summary_solving["total_zones"] * 100).round(2)
summary_solving["pct_after_exp_with_wait"] = (summary_solving["solved_after_expansion_with_waiting"] / summary_solving["total_zones"] * 100).round(2)
summary_solving["pct_unsolved"] = (summary_solving["unsolved"] / summary_solving["total_zones"] * 100).round(2)

summary_solving_row = summary_solving.to_frame().T
summary_solving_row.index = ["ALL"]

solving_methods_with_total = pd.concat([solving_methods_stats, summary_solving_row], axis=0)

print("\n\nLNS: Zone solving methods breakdown per agent count")
print("Categories:")
print("  - solved_no_expansion_no_waiting: Solved on first attempt, no extension, no waiting")
print("  - solved_no_expansion_with_waiting: Solved without extension but used waiting time")
print("  - solved_after_expansion_no_waiting: Extended and solved without waiting (on first extension attempt)")
print("  - solved_after_expansion_with_waiting: Extended and solved using waiting time")
print("  - unsolved: Never solved")
display(solving_methods_with_total)

# Verification: Check if we're correctly identifying final states
print("\n\nVerification: Checking for zones with mixed solved/unsolved states")
zone_groups_all = identify_unique_zones(lns_local_zones)
zones_with_mixed_states = []
zones_with_only_unsolved = []

for zone_group in zone_groups_all:
    solved_values = zone_group["solved"].unique()
    max_attempt = zone_group["zone_attempt"].max()
    final_state = zone_group[zone_group["zone_attempt"] == max_attempt].iloc[0]
    
    # Get original zone boundaries (from zone_attempt=0)
    orig_row = zone_group[zone_group["zone_attempt"] == 0].iloc[0]
    exp_id = orig_row["experiment_id"]
    makespan = orig_row["makespan_attempt"]
    start_t = orig_row["start_t"]
    end_t = orig_row["end_t"]
    
    if len(solved_values) > 1:
        # Zone has both solved=0 and solved=1 rows
        zones_with_mixed_states.append({
            "experiment_id": exp_id,
            "makespan_attempt": makespan,
            "start_t": start_t,
            "end_t": end_t,
            "agent_count": final_state["agent_count"],
            "max_zone_attempt": max_attempt,
            "final_solved": final_state["solved"],
            "num_attempts": len(zone_group),
            "solved_values": str(solved_values),
        })
    elif solved_values[0] == 0:
        # Zone has only unsolved attempts
        zones_with_only_unsolved.append({
            "experiment_id": exp_id,
            "makespan_attempt": makespan,
            "start_t": start_t,
            "end_t": end_t,
            "agent_count": final_state["agent_count"],
            "max_zone_attempt": max_attempt,
            "num_attempts": len(zone_group),
        })

print(f"Zones with mixed solved/unsolved states: {len(zones_with_mixed_states)}")
if len(zones_with_mixed_states) > 0:
    print("Sample zones with mixed states (should all have final_solved=1):")
    mixed_df = pd.DataFrame(zones_with_mixed_states)
    display(mixed_df.head(10))
    print(f"\nOf {len(zones_with_mixed_states)} zones with mixed states:")
    print(f"  - Final state solved=1: {(mixed_df['final_solved'] == 1).sum()}")
    print(f"  - Final state solved=0: {(mixed_df['final_solved'] == 0).sum()}")

print(f"\nZones with only unsolved attempts: {len(zones_with_only_unsolved)}")
if len(zones_with_only_unsolved) > 0:
    unsolved_df = pd.DataFrame(zones_with_only_unsolved)
    print("Unsolved zones by agent count:")
    display(unsolved_df.groupby("agent_count").size().reset_index(name="unsolved_zones"))
    print("\nUnsolved zones by number of attempts:")
    display(unsolved_df.groupby("num_attempts").size().reset_index(name="count"))


Found 2624 concluded experiments in experiments.csv files
Filtered local zones: 63572 total rows -> 62560 rows from concluded experiments
Excluded 1012 zone rows from unfinished experiments
LNS: Zone metrics (conflicts and positions) per agent count


,conflicts_mean,conflicts_median,conflicts_std,conflicts_min,conflicts_max,positions_mean,positions_median,positions_std,positions_min,positions_max
agent_count,,,,,,,,,,
10,1.34,1.00,1.39,1,30,33.08,8.00,93.12,4,682
20,1.52,1.00,2.89,1,118,12.80,5.00,37.39,4,584
50,1.55,1.00,3.97,1,126,8.31,5.00,16.12,4,284
100,1.33,1.00,3.27,1,399,6.83,5.00,6.12,4,275
200,1.39,1.00,2.91,1,121,6.89,5.00,7.64,4,267



LNS: Zone metrics (conflicts and positions) per agent count (with ALL summary row)


,conflicts_mean,conflicts_median,conflicts_std,conflicts_min,conflicts_max,positions_mean,positions_median,positions_std,positions_min,positions_max
10,1.34,1.00,1.39,1.00,30.00,33.08,8.00,93.12,4.00,682.00
20,1.52,1.00,2.89,1.00,118.00,12.80,5.00,37.39,4.00,584.00
50,1.55,1.00,3.97,1.00,126.00,8.31,5.00,16.12,4.00,284.00
100,1.33,1.00,3.27,1.00,399.00,6.83,5.00,6.12,4.00,275.00
200,1.39,1.00,2.91,1.00,121.00,6.89,5.00,7.64,4.00,267.00
ALL,1.40,1.00,3.26,1.00,399.00,8.31,5.00,21.41,4.00,682.00



LNS: Zone building statistics per agent count (with ALL summary row)


...

,experiment_id,makespan_attempt,start_t,end_t,agent_count,max_zone_attempt,final_solved,num_attempts,solved_values
0,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,0,112,114,10,1,1,2,[0 1]
1,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,0,102,104,10,1,1,2,[0 1]
2,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,0,179,181,10,1,1,2,[0 1]
3,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,0,170,172,10,1,1,2,[0 1]
4,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,0,111,113,10,1,1,2,[0 1]
5,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,0,104,106,10,1,1,2,[0 1]
6,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,0,160,162,10,1,1,2,[0 1]
7,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,0,21,23,100,1,1,2,[0 1]
8,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,0,108,110,100,1,1,2,[0 1]
9,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,0,120,122,100,1,1,2,[0 1]



Of 9461 zones with mixed states:
  - Final state solved=1: 9461
  - Final state solved=0: 0

Zones with only unsolved attempts: 3
Unsolved zones by agent count:


,agent_count,unsolved_zones
0,10,2
1,50,1



Unsolved zones by number of attempts:


,num_attempts,count
0,8,1
1,37,1
2,40,1


In [198]:
# Quick check: Show examples of unsolved zones
def find_unsolved_zones(zones_df: pd.DataFrame, n_examples: int = 5):
    """Find and display examples of zones that were never solved."""
    
    # Use the helper function to identify unique zones by original boundaries
    zone_groups = identify_unique_zones(zones_df)
    
    unsolved_zones = []
    for zone_group in zone_groups:
        # Find final state (highest zone_attempt)
        max_attempt_idx = zone_group["zone_attempt"].idxmax()
        final_state = zone_group.loc[max_attempt_idx]
        
        if final_state["solved"] == 0:
            # This zone was never solved
            # Get original zone boundaries (from zone_attempt=0)
            orig_row = zone_group[zone_group["zone_attempt"] == 0].iloc[0]
            unsolved_zones.append({
                "experiment_id": orig_row["experiment_id"],
                "makespan_attempt": orig_row["makespan_attempt"],
                "start_t": orig_row["start_t"],
                "end_t": orig_row["end_t"],
                "agent_count": final_state["agent_count"],
                "max_zone_attempt": final_state["zone_attempt"],
                "num_attempts": len(zone_group),
                "conflicts": final_state["conflicts"],
                "zone_positions": final_state["zone_positions"],
            })
    
    if unsolved_zones:
        unsolved_df = pd.DataFrame(unsolved_zones)
        print(f"Found {len(unsolved_zones)} zones that were never solved")
        print(f"\nShowing first {min(n_examples, len(unsolved_zones))} examples:")
        display(unsolved_df.head(n_examples))
        
        # Summary by agent count
        print(f"\nUnsolved zones by agent count:")
        display(unsolved_df.groupby("agent_count").size().reset_index(name="unsolved_zones"))
    else:
        print("No unsolved zones found - all zones were eventually solved!")


find_unsolved_zones(lns_local_zones, n_examples=10)


Found 3 zones that were never solved

Showing first 3 examples:


,experiment_id,makespan_attempt,start_t,end_t,agent_count,max_zone_attempt,num_attempts,conflicts,zone_positions
0,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,0,3,12,50,7,8,52,256
1,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,0,26,28,10,36,37,1,682
2,_home_max_Documents_PhD_MAPF_LNS_MAPF_LNS_SAT_...,0,37,41,10,39,40,11,682



Unsolved zones by agent count:


,agent_count,unsolved_zones
0,10,2
1,50,1


In [199]:
# Makespan increase analysis for LNS

def summarize_makespan_increases(attempts_df: pd.DataFrame) -> pd.DataFrame:
    """
    For each experiment, look at the maximum makespan_attempt.
    - 0  -> no makespan increase
    - k>0 -> k makespan increases in that experiment.
    Summarize by agent_count.
    """
    per_experiment = (
        attempts_df
        .groupby(["experiment_id", "agent_count"], as_index=False)["makespan_attempt"]
        .max()
        .rename(columns={"makespan_attempt": "max_makespan_attempt"})
    )

    # Overall counts by agent_count
    grouped = per_experiment.groupby("agent_count")
    summary = grouped.agg(
        num_experiments=("experiment_id", "nunique"),
        experiments_with_increase=("max_makespan_attempt", lambda s: (s > 0).sum()),
        total_increases=("max_makespan_attempt", "sum"),
    )

    summary["pct_experiments_with_increase"] = (
        summary["experiments_with_increase"] / summary["num_experiments"] * 100
    ).round(2)

    return summary.sort_index()


# Reuse lns_attempts from the makespan_attempts analysis; reload if needed
try:
    _ = lns_attempts
except NameError:
    lns_attempts = load_lns_makespan_attempts()

makespan_increase_stats = summarize_makespan_increases(lns_attempts)

print("LNS: makespan increase statistics by agent count")
display(makespan_increase_stats)

LNS: makespan increase statistics by agent count


,num_experiments,experiments_with_increase,total_increases,pct_experiments_with_increase
agent_count,,,,
10,923,2,2,0.22
20,751,0,0,0.00
50,566,1,1,0.18
100,316,0,0,0.00
200,68,0,0,0.00


In [200]:
around 30 % of zones would have required expansion but were solvable by slack allocation.
H


SyntaxError: invalid syntax (3607530027.py, line 1)

In [ ]:
Slack distribution improvement could target agents involved in more conflicts, to perserve overall slack use
agents with little or no slack to begin with can maybe get priority routing in the initial solution planning phase. And they could be preferred to not have to use their slack.

In [ ]:
for practicality the makespan slack can be altered
defining individual target times, incase of unsovle everyone or single agents can get more slack